# F3 — LangChain & LangGraph · Hands-On Lab Notebook

**AI Engineering Specialist Track · Foundation Module F3**

Three labs, ~65 minutes, fully runnable offline:

1. **LangChain pipeline (LCEL)** — classify customer complaints by *category, sentiment, and priority* using `prompt | model | parser` and a Pydantic schema.
2. **LangGraph workflow** — a stateful triage graph that **branches** to FAQ-bot, knowledge-base RAG, or a human queue based on classifier confidence.
3. **LangSmith-style tracing** — instrument both apps, inspect a multi-step run, and find the slowest node.

---

### A note on how this notebook runs

The labs teach the **real** LangChain and LangGraph patterns — the LCEL pipe
(`prompt | model | parser`), `StateGraph` with conditional edges, and trace spans.

To guarantee it runs **with zero errors and zero warnings on any machine**, it uses
small, self-contained stand-ins for the LLM and the two frameworks instead of the heavy
library stack. Every line maps 1:1 to the real API — the *Case Analysis Solutions* doc
shows the exact real-library equivalent for each lab. In production you swap the engine,
not your code.

## Setup

Only the standard scientific-Python stack plus `pydantic` — nothing that downloads
models or needs an API key.

In [ ]:
# Clean, quiet environment ----------------------------------------------------
import os, warnings, logging, time, json, re, random
from dataclasses import dataclass, field
from typing import Callable, Any, Literal

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
logging.getLogger("matplotlib").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field, field_validator

random.seed(42)
np.random.seed(42)
print("Environment ready.")
print("Pydantic:", __import__("pydantic").VERSION, "| Pandas:", pd.__version__)

## Shared building blocks — a mini LCEL runtime

LangChain's headline idea is **LCEL**: compose small units with the `|` pipe, e.g.
`prompt | model | parser`. Each unit is a **Runnable** with `.invoke()`. We implement
that contract so the rest of the notebook reads like real LangChain.

In [ ]:
class Runnable:
    """Minimal LCEL-style Runnable. Real LangChain Runnables share this contract:
    an .invoke(x) method and a | operator that chains them left-to-right."""
    def __init__(self, fn: Callable[[Any], Any], name: str = "runnable"):
        self.fn = fn
        self.name = name

    def invoke(self, x):
        return self.fn(x)

    def __or__(self, other: "Runnable") -> "Runnable":
        def composed(x, _s=self, _o=other):
            return _o.fn(_s.fn(x))
        return Runnable(composed, name=f"{self.name}|{other.name}")

def as_runnable(name):
    def wrap(fn):
        return Runnable(fn, name=name)
    return wrap

print("Runnable ready — supports .invoke() and the | pipe, just like LCEL.")

### A deterministic mock chat model

In real LangChain this slot is `ChatAnthropic(...)` or `ChatOpenAI(...)`. Here it's a
deterministic rules-based stand-in returning a JSON string — the shape a real chat
model's `.content` carries when you ask for structured output.

In [ ]:
_CATEGORY_RULES = [
    ("billing",  ["bill", "charge", "charged", "refund", "payment", "invoice", "overcharg"]),
    ("network",  ["signal", "network", "internet", "connection", "slow", "outage", "drop"]),
    ("device",   ["phone", "device", "sim", "handset", "router", "modem", "screen", "pixel"]),
    ("account",  ["login", "log in", "password", "account", "profile", "locked", "verify", "access"]),
    ("support",  ["agent", "rude", "waited", "hold", "support", "service", "unhelpful"]),
]
_POSITIVE = ["love", "great", "excellent", "happy", "thank", "perfect", "best", "quickly"]
_NEGATIVE = ["angry", "terrible", "worst", "furious", "unacceptable", "frustrat", "rude", "horrible"]
_URGENT   = ["urgent", "immediately", "asap", "emergency", "cancel", "legal"]

def _score_category(text):
    t = text.lower()
    best, best_hits = "other", 0
    for cat, kws in _CATEGORY_RULES:
        hits = sum(t.count(k) for k in kws)
        if hits > best_hits:
            best, best_hits = cat, hits
    conf = 0.55 + 0.43 * (1 - np.exp(-best_hits)) if best_hits else 0.40
    return best, round(float(conf), 2)

def _score_sentiment(text):
    t = text.lower()
    pos = sum(t.count(k) for k in _POSITIVE)
    neg = sum(t.count(k) for k in _NEGATIVE)
    if pos > neg: return "positive"
    if neg > pos: return "negative"
    return "neutral"

def _score_priority(text):
    t = text.lower()
    urgent = sum(t.count(k) for k in _URGENT)
    neg    = sum(t.count(k) for k in _NEGATIVE)
    if urgent >= 1 or neg >= 2: return "high"
    if neg == 1:                return "medium"
    return "low"

def mock_chat_model(prompt_text: str) -> str:
    """Stand-in for ChatAnthropic/ChatOpenAI. Returns a JSON string (like .content)."""
    marker = "TICKET:"
    text = prompt_text.split(marker, 1)[1].strip() if marker in prompt_text else prompt_text
    category, confidence = _score_category(text)
    return json.dumps({
        "category":   category,
        "sentiment":  _score_sentiment(text),
        "priority":   _score_priority(text),
        "confidence": confidence,
    })

print("Mock chat model ready (deterministic; returns JSON like a real .content field).")

---
# Lab 1 — A LangChain pipeline (LCEL) for complaint classification

**Goal:** build `prompt | model | parser` that turns a raw complaint into a validated
`Ticket` object with *category, sentiment, priority, confidence*.

### Step 1.1 — The Pydantic output schema (the contract)

### Step 1.2 — Prompt, model, and parser as Runnables

In real LangChain: `ChatPromptTemplate`, a chat model, and `PydanticOutputParser`.
Here each is a `Runnable`, so we pipe them with `|`.

### Step 1.3 — Run the chain on a single ticket

### Step 1.4 — Batch the chain over a corpus

In [ ]:
COMPLAINTS = [
    "I was charged twice on my bill this month and the agent was rude!",
    "No internet signal in my area since yesterday, this is urgent.",
    "My new phone screen has a dead pixel right out of the box.",
    "I cannot log in, my account is locked and I need access immediately.",
    "The support team was excellent and resolved my issue quickly, thank you!",
    "Bill went up 40% with no warning and no clear explanation.",
    "Internet keeps dropping every few minutes during calls.",
    "I want to cancel my account, the service has been terrible.",
    "Password reset email never arrives, I am completely locked out.",
    "Router keeps overheating and restarting on its own.",
    "Refund still not processed after three weeks, this is unacceptable.",
    "Great coverage now in my neighbourhood, very happy with the upgrade.",
]

rows = []
for text in COMPLAINTS:
    t = classify_chain.invoke(text)
    rows.append({**t.model_dump(), "text": text})

df_tickets = pd.DataFrame(rows)
df_tickets[["category", "sentiment", "priority", "confidence", "text"]]

### Step 1.5 — Classification distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, col, color in zip(axes, ["category", "sentiment", "priority"],
                          ["#10B981", "#0EA5E9", "#D97706"]):
    counts = df_tickets[col].value_counts()
    ax.bar(counts.index, counts.values, color=color)
    ax.set_title(col.capitalize(), fontsize=12, fontweight="bold")
    ax.tick_params(axis="x", rotation=30)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
fig.suptitle("Lab 1 — complaint classification (LCEL pipeline)", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

### Lab 1 — What to look for (discussion)

1. **The `|` pipe is the whole idea.** `prompt | model | parser` is real LCEL. Each stage is a Runnable with `.invoke()`; composition is type-checked hand-offs.
2. **The parser is the guardrail.** `PydanticOutputParser` means a malformed or out-of-enum field *raises* instead of slipping downstream.
3. **Confidence drives Lab 2.** We route on the `confidence` field: high-confidence to automation, low to a human.
4. **Why LCEL over hand-rolled glue?** Streaming, batching, retries, and async come free on real Runnables; your composition stays identical.

---
# Lab 2 — A LangGraph triage workflow with conditional routing

**Goal:** a **stateful graph** that classifies a ticket, then **branches**:

* **high confidence + low priority**  → **FAQ-bot**
* **high confidence + high priority**  → **knowledge-base RAG**
* **low confidence**                   → **human queue**

### Step 2.1 — A mini StateGraph engine

Mirrors LangGraph's real API: `add_node`, `add_edge`, `add_conditional_edges`,
`set_entry_point`, `compile`, `invoke`.

In [ ]:
START, END = "__start__", "__end__"

class StateGraph:
    """Minimal LangGraph-style state machine. Real LangGraph shares this API surface:
    nodes are functions state->state; conditional edges choose the next node at runtime."""
    def __init__(self):
        self.nodes, self.edges, self.cond_edges, self.entry = {}, {}, {}, None
    def add_node(self, name, fn):                 self.nodes[name] = fn
    def add_edge(self, src, dst):                 self.edges[src] = dst
    def add_conditional_edges(self, src, router): self.cond_edges[src] = router
    def set_entry_point(self, name):              self.entry = name
    def compile(self):                            return self
    def invoke(self, state, trace=None):
        node, steps = self.entry, 0
        while node not in (END, None) and steps < 100:
            t0 = time.perf_counter()
            state = self.nodes[node](dict(state))
            if trace is not None:
                trace.record(node, (time.perf_counter() - t0) * 1000)
            steps += 1
            if node in self.cond_edges:  node = self.cond_edges[node](state)
            elif node in self.edges:     node = self.edges[node]
            else:                        node = END
        return state

print("StateGraph engine ready (add_node / add_edge / add_conditional_edges / compile / invoke).")

### Step 2.2 — Define the nodes

### Step 2.3 — The conditional router

Reads the post-classification state and returns the **name of the next node** — the
callable you pass to `add_conditional_edges` in real LangGraph.

### Step 2.4 — Wire and compile the graph

### Step 2.5 — Run tickets through the graph and inspect routing

In [ ]:
routed = []
for text in COMPLAINTS:
    final = triage_app.invoke({"text": text})
    routed.append({
        "handler":    final["handler"],
        "category":   final["ticket"]["category"],
        "priority":   final["ticket"]["priority"],
        "confidence": final["ticket"]["confidence"],
        "text":       text,
    })

df_routed = pd.DataFrame(routed)
df_routed[["handler", "category", "priority", "confidence", "text"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
counts = df_routed["handler"].value_counts()
colors = {"faq_bot": "#10B981", "kb_rag": "#0EA5E9", "human_queue": "#D97706"}
ax.bar(counts.index, counts.values, color=[colors.get(h, "#64748B") for h in counts.index])
ax.set_title("Lab 2 — where the triage graph routed each ticket", fontsize=12, fontweight="bold")
ax.set_ylabel("tickets")
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.tight_layout()
plt.show()

print("Routing summary:")
for handler, n in counts.items():
    print(f"  {handler:<14} {n} ticket(s)")

### Lab 2 — What to look for (discussion)

1. **Conditional edges are why LangGraph exists.** A plain chain is a straight line; the moment you need branching, retries, or human-in-the-loop you want a graph. `route_after_classify` is the branch.
2. **State is shared and explicit.** Every node receives and returns the full state dict — that visibility is what makes graphs debuggable and **checkpointable**.
3. **The threshold is a product decision.** Moving `CONFIDENCE_THRESHOLD` trades automation rate against escalation safety; watch the routing chart shift.
4. **Each terminal node is a different cost/quality point.** FAQ-bot cheapest, RAG grounded-but-costlier, human most expensive — the graph routes spend to where it's justified.

---
# Lab 3 — LangSmith-style tracing & observability

**Goal:** instrument the triage graph so every node emits a **span** (name + latency),
assemble spans into traces, inspect one run, and find the **slowest node** across runs.

### Step 3.1 — A minimal tracer (the callback that records spans)

### Step 3.2 — Re-run the graph with tracing enabled

A small deterministic per-node delay makes the latency comparison meaningful and stable
(RAG heaviest, human lightest).

In [ ]:
_NODE_DELAY_MS = {"classify": 6.0, "faq": 3.0, "rag": 12.0, "human": 1.0}

def _instrument(fn, name):
    def wrapped(state):
        time.sleep(_NODE_DELAY_MS[name] / 1000.0)   # simulate node work
        return fn(state)
    return wrapped

tg = StateGraph()
tg.add_node("classify", _instrument(classify_node, "classify"))
tg.add_node("faq",      _instrument(faq_node, "faq"))
tg.add_node("rag",      _instrument(rag_node, "rag"))
tg.add_node("human",    _instrument(human_node, "human"))
tg.set_entry_point("classify")
tg.add_conditional_edges("classify", route_after_classify)
tg.add_edge("faq", END); tg.add_edge("rag", END); tg.add_edge("human", END)
traced_app = tg.compile()

traces = []
for i, text in enumerate(COMPLAINTS):
    tr = Trace(run_id=i)
    traced_app.invoke({"text": text}, trace=tr)
    traces.append(tr)

print(f"Collected {len(traces)} traces.")

### Step 3.3 — Inspect one multi-step run (a single trace)

### Step 3.4 — Aggregate across runs to find the slowest node

In [ ]:
span_rows = [{"node": n, "latency_ms": ms} for tr in traces for n, ms in tr.spans]
df_spans = pd.DataFrame(span_rows)
node_stats = (df_spans.groupby("node")["latency_ms"]
              .agg(calls="count", mean_ms="mean",
                   p95_ms=lambda s: np.percentile(s, 95), total_ms="sum")
              .sort_values("total_ms", ascending=False)
              .round(2))
print("Per-node latency profile (sorted by total time spent):")
print(node_stats)

slowest = node_stats["mean_ms"].idxmax()
print(f"\nSlowest node by mean latency: '{slowest}'  "
      f"({node_stats.loc[slowest, 'mean_ms']} ms avg)")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
order = node_stats.sort_values("mean_ms")
ax.barh(order.index, order["mean_ms"], color="#10B981")
ax.set_title("Lab 3 — mean latency per node (find the bottleneck)", fontsize=12, fontweight="bold")
ax.set_xlabel("mean latency (ms)")
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
fig.tight_layout()
plt.show()

### Lab 3 — What to look for (discussion)

1. **Traces turn a black box into a profiler.** Spans tell you *which node* is slow — here `rag` dominates, exactly where you'd focus optimisation.
2. **`p95`, not just `mean`.** Real traffic has tails; p95 is what users feel on a bad run.
3. **This is how you justify architecture changes.** "RAG is 4× FAQ-bot latency and serves 25% of tickets" is a data-backed case for caching the RAG path.
4. **In production this is LangSmith.** Same model: every Runnable and node recorded with timing and I/O; you instrument once and observe everything.

---
# F3 Lab — Wrap-up checklist

- [ ] What does the LCEL `|` pipe do, and what contract makes it work?
- [ ] Where does the output parser sit in `prompt | model | parser`, and what does it protect against?
- [ ] What does a LangGraph `StateGraph` give you that a straight chain does not?
- [ ] What is a *conditional edge*, and what decided the route in Lab 2?
- [ ] Why is shared explicit state the prerequisite for checkpointing and human-in-the-loop?
- [ ] In tracing, why read `p95` alongside `mean`?
- [ ] Which node was the bottleneck in Lab 3, and what would you try first?

**Next:** F4 — Retrieval-Augmented Generation. The `rag` node you routed to in Lab 2 becomes the whole subject.